# 🌾 Minería: segmentación de productores — CRISP-DM
**Gravetal Bolivia S.A.** · No supervisado sobre Oro (`FACT_RECEPCION_GRANO` + `DIM_PRODUCTOR`, 1,897 productores) con **k-means** (segmentos) y **DBSCAN** (atípicos). Corre en Lightning.ai con `data/productores_features.csv` o en vivo contra Snowflake.

| Fase CRISP-DM | Criterios |
|---|---|
| **1. Negocio** | 6. Importancia |
| **2. Datos** | 1. Dataset · 10. Nube · 11. Variables · 8. EDA |
| **3. Preparación** | 2-3. Partición de estabilidad |
| **4. Modelado** | 5. Modelo |
| **5. Evaluación** | 4. Silueta/Davies-Bouldin · 9. Estabilidad · 7-8. Resultados y visualización |
| **6. Despliegue** | 10. Tabla de segmentos + monitoreo |

In [ ]:
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score
from sklearn.decomposition import PCA
import sklearn, matplotlib
print('sklearn', sklearn.__version__, '| pandas', pd.__version__)
sns.set_theme(style='whitegrid', palette='muted')
SEED = 42

## 1. Negocio · Criterio 6
**Problema:** se paga y compra igual a 1,897 productores con mermas y calidades distintas. **Minería:** agruparlos por volumen, frecuencia, merma y calidad para precio diferencial, visitas técnicas y cupos. **Éxito:** clusters estables (ARI≥0.60) e interpretables que cambien una decisión por grupo. Lección v1: K=2 con silueta 0.964 aislaba un registro imposible (ID 525: 4,381M TN); se cuarentena y se modela con dispersión (std, max, % eventos críticos).

## 2. Datos · Criterios 1, 10, 11

In [ ]:
CSV = 'data/productores_features_v2.csv'
df = None
try:
    import snowflake.connector
    c = snowflake.connector.connect(user=os.getenv('SNOWFLAKE_USER', 'ENRRIQUE'),
        password=os.environ['SNOWFLAKE_PASSWORD'], account=os.getenv('SNOWFLAKE_ACCOUNT', 'AVBVHGL-WZ57062'),
        database='AIRBYTE_DATABASE', warehouse='COMPUTE_WH')
    q = open('sql/productores_features.sql', encoding='utf-8').read()
    cur = c.cursor(); cur.execute(q)
    df = pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
    print('Origen: Snowflake en vivo,', len(df), 'productores')
except Exception as e:
    print('Snowflake no disponible:', type(e).__name__, '-> uso CSV local')
    df = pd.read_csv(CSV)
    print('Origen: CSV,', len(df), 'productores')
df.columns = df.columns.str.lower()
display(df.head())
print('Nulos:'); print(df.isna().sum().to_string())

**Variables (Criterio 11):** `n_recepciones, volumen_total_tn, ticket_medio_tn, merma_total_tn, humedad_media, impurezas_media, proteina_media, aceite_media, pct_descuento, n_silos` + descriptivas (`tipo_productor, municipio, area_total_ha`). Target no hay (no supervisado): el 'acierto' es estabilidad + silueta + sentido de negocio.

In [ ]:
# EDA · Criterio 8
print(df.describe().T.round(2).to_string())
num = ['n_recepciones','volumen_total_tn','merma_total_tn','humedad_media','humedad_std','pct_hum_crit','impurezas_media','impurezas_std','pct_imp_crit','pct_descuento','max_merma_evento']
fig, ax = plt.subplots(2, 2, figsize=(14, 8))
df['volumen_total_tn'].hist(bins=40, ax=ax[0,0]); ax[0,0].set_title('Volumen total por productor (sesgo?)')
sns.scatterplot(data=df, x='humedad_media', y='impurezas_media', size='volumen_total_tn', alpha=.5, ax=ax[0,1]); ax[0,1].set_title('Humedad vs impurezas')
sns.heatmap(df[num].corr(), annot=False, cmap='coolwarm', ax=ax[1,0]); ax[1,0].set_title('Correlación (multicolinealidad)')
df['tipo_productor'].value_counts().plot.bar(ax=ax[1,1]); ax[1,1].set_title('Tipo de productor')
plt.tight_layout(); plt.show()

## 3. Preparación · Criterios 2-3 adaptados
Sin train/test clásico: se escala (log a sesgadas + StandardScaler) y se reserva una mitad para probar **estabilidad** de los clusters.

In [ ]:
FEATS = ['n_recepciones','volumen_total_tn','ticket_medio_tn','merma_total_tn','humedad_media','humedad_std','humedad_max','pct_hum_crit','impurezas_media','impurezas_std','impurezas_max','pct_imp_crit','proteina_media','aceite_media','pct_descuento','max_merma_evento']
d = df[(df['id_productor'] != 525)].dropna(subset=FEATS).copy()  # cuarentena 525 (registro imposible)
print('Filas útiles (sin 525):', len(d))
X = np.log1p(d[FEATS].clip(lower=0))
scaler = StandardScaler()
Xs = scaler.fit_transform(X)
print('Media~0:', np.abs(Xs.mean()).max().round(4))

## 4. Modelado · Criterio 5
**k-means** (k=2..8, codo + silueta) para segmentos accionables; **DBSCAN** (ε por k-distancia) para atípicos que k-means escondería. Baseline: 1 solo grupo.

In [ ]:
res = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit(Xs)
    res[k] = {'silhouette': silhouette_score(Xs, km.labels_), 'davies_bouldin': davies_bouldin_score(Xs, km.labels_),
            'calinski': calinski_harabasz_score(Xs, km.labels_)}
lb = pd.DataFrame(res).T.round(3).sort_values('silhouette', ascending=False)
print('LEADERBOARD k-means'); display(lb)
K = int(lb.index[0])
print('K elegido por silueta:', K)

In [ ]:
# DBSCAN: eps por k-distancia (k=2*dim) · Criterio 5
from sklearn.neighbors import NearestNeighbors
kdist = NearestNeighbors(n_neighbors=2*len(FEATS)).fit(Xs)
dist, _ = kdist.kneighbors(Xs)
knee = np.sort(dist[:, -1])
EPS = float(np.quantile(knee, 0.95))
db = DBSCAN(eps=EPS, min_samples=2*len(FEATS)).fit(Xs)
n_cl = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
print(f'DBSCAN eps={EPS:.3f}: {n_cl} grupos + {(db.labels_==-1).sum()} atípicos ({(db.labels_==-1).mean():.1%})')

## 5. Evaluación · Criterios 4, 9, 7, 8

In [ ]:
final = KMeans(n_clusters=K, n_init=50, random_state=SEED).fit(Xs)
d['segmento'] = final.labels_
# Estabilidad split-half · Criterio 9
a, b = np.array_split(np.random.RandomState(SEED).permutation(len(Xs)), 2)
la = KMeans(n_clusters=K, n_init=20, random_state=1).fit_predict(Xs[a])
lb2 = KMeans(n_clusters=K, n_init=20, random_state=2).fit_predict(Xs[b])
ari = adjusted_rand_score(KMeans(n_clusters=K, n_init=20, random_state=1).fit_predict(Xs), KMeans(n_clusters=K, n_init=20, random_state=2).fit_predict(Xs))
print(f'Silueta={silhouette_score(Xs, final.labels_):.3f} DB={davies_bouldin_score(Xs, final.labels_):.3f} ARI-estabilidad={ari:.3f} -> {"CUMPLE ARI>=0.60" if ari>=0.60 else "IN ESTABLE"}')
print('Tamaños:', pd.Series(final.labels_).value_counts().sort_index().to_dict())

In [ ]:
# Perfiles por segmento · Criterio 7
prof = d.groupby('segmento')[FEATS].mean().round(2)
prof['n'] = d.groupby('segmento').size()
display(prof.sort_values('volumen_total_tn', ascending=False))
print('Atípicos DBSCAN (revisar uno por uno):', int((db.labels_==-1).sum()))

In [ ]:
# Visualización · Criterio 8
pca = PCA(n_components=2, random_state=SEED).fit_transform(Xs)
fig, ax = plt.subplots(1, 3, figsize=(17, 5))
ax[0].plot(list(res), [res[k]['silhouette'] for k in res], 'o-'); ax[0].set_title('Codo de silueta'); ax[0].set_xlabel('k')
sc = ax[1].scatter(pca[:,0], pca[:,1], c=final.labels_, cmap='tab10', alpha=.5, s=12); ax[1].set_title(f'k-means K={K} (PCA)')
ax[2].scatter(pca[:,0], pca[:,1], c=(db.labels_==-1), cmap='coolwarm', alpha=.5, s=12); ax[2].set_title('DBSCAN: atípicos en rojo')
plt.tight_layout(); plt.show()

## 6. Despliegue · Criterio 10
Se guarda `data/productores_segmentos.csv` (productor → segmento + flag atípico). Escritura a `ORO.MINERIA_SEGMENTO_PRODUCTOR` opcional. **Monitoreo:** re-ejecutar por campaña; si ARI < 0.50 o un segmento cambia >30% de tamaño, redefinir.

In [ ]:
out = d[['id_productor','nombre_completo','tipo_productor','municipio','segmento']].copy()
out['atipico_dbscan'] = (db.labels_ == -1).astype(int)
out.to_csv('data/productores_segmentos_v2.csv', index=False, encoding='utf-8')
print('Guardado data/productores_segmentos_v2.csv:', len(out), 'filas')
ESCRIBIR_SNOWFLAKE = False
print('Para Snowflake: SNOWFLAKE_PASSWORD por secreto/entorno + CREATE TABLE ORO.MINERIA_SEGMENTO_PRODUCTOR.')